In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, default_data_collator
from sklearn.metrics import fbeta_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, multilabel_confusion_matrix

# LOAD MODEL AND INITIALIZE TRAINER

model_path = "./.saved_models/UpdatedTwitterRoBERTaV2" 

# Load using the Auto classes so the vocabulary maps perfectly
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(
    model_path, 
    problem_type="multi_label_classification"
)

# Initialize a basic Trainer just for generating predictions
trainer = Trainer(
    model=model,
    data_collator=default_data_collator
)

# LOAD GOEMOTIONS DATASET
goEmotionsDataset = load_dataset('go_emotions')
sentimentLabels = goEmotionsDataset['train'].features['labels'].feature.names
num_labels = len(sentimentLabels)

def tokenize_and_encode(batch):
    tokenized = tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )
    
    # Ensure this is a list of float32 arrays (multi-label multi-hot encoding)
    batch_labels = []   
    for labels_list in batch["labels"]:
        encoded = np.zeros(num_labels, dtype=np.float32)
        encoded[labels_list] = 1.0
        batch_labels.append(encoded)
        
    tokenized["labels"] = batch_labels
    return tokenized

print("Tokenizing Validation and Test sets...")
# We use the validation set to calculate thresholds
val_dataset = goEmotionsDataset['validation'].map(tokenize_and_encode, batched=True, remove_columns=["text", "id"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# We use the test set for the final evaluation
test_dataset = goEmotionsDataset['test'].map(tokenize_and_encode, batched=True, remove_columns=["text", "id"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
print("Done!")



# THRESHOLDS
def optimize_thresholds(trainer, eval_dataset):
    print("Generating predictions to calculate F0.5 thresholds...")

    # Remove labels temporarily to avoid loss computation
    eval_dataset_no_labels = eval_dataset.remove_columns("labels")

    predictions = trainer.predict(eval_dataset_no_labels)

    logits = predictions.predictions
    if isinstance(logits, tuple):
        logits = logits[0]

    true_labels = np.array(eval_dataset["labels"])

    # Apply Sigmoid to get probabilities
    probs = 1 / (1 + np.exp(-logits))

    best_thresholds = []

    for i in range(num_labels):
        best_t = 0.5
        best_fscore = 0.0

        for t in np.arange(0.1, 0.95, 0.05):
            preds = (probs[:, i] >= t).astype(int)
            score = fbeta_score(true_labels[:, i], preds, beta=0.5, zero_division=0)

            if score > best_fscore:
                best_fscore = score
                best_t = t

        best_thresholds.append(best_t)

    return np.array(best_thresholds)

optimal_thresholds = optimize_thresholds(trainer, val_dataset)

print("\n--- Optimal Thresholds Per Class (F0.5) ---")
for name, thresh in zip(sentimentLabels, optimal_thresholds):
    print(f"{name.capitalize():<15}: {thresh:.2f}")


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2255.09it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              


Tokenizing Validation and Test sets...


Map: 100%|██████████| 5427/5427 [00:00<00:00, 37044.88 examples/s]
/Users/donovanchen/UCI/CS175/RedditSentimentAnalysis/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Done!
Generating predictions to calculate F0.5 thresholds...



--- Optimal Thresholds Per Class (F0.5) ---
Admiration     : 0.65
Amusement      : 0.65
Anger          : 0.55
Annoyance      : 0.50
Approval       : 0.50
Caring         : 0.60
Confusion      : 0.65
Curiosity      : 0.55
Desire         : 0.50
Disappointment : 0.55
Disapproval    : 0.60
Disgust        : 0.70
Embarrassment  : 0.70
Excitement     : 0.55
Fear           : 0.50
Gratitude      : 0.70
Grief          : 0.70
Joy            : 0.60
Love           : 0.75
Nervousness    : 0.65
Optimism       : 0.60
Pride          : 0.50
Realization    : 0.65
Relief         : 0.20
Remorse        : 0.55
Sadness        : 0.70
Surprise       : 0.65
Neutral        : 0.50


In [5]:
# LOAD POLITICAL MODEL

In [ ]:
# LOAD TOPIC MODEL

In [ ]:
# RUN ON TEST SET

In [ ]:
# EVALUATE